In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -709, 100000)))

In [ ]:
# INPUT_DIR = Path('../data/g2net/') # my env
INPUT_DIR = Path('input/')
RESULT_DIR = Path('results/')

In [ ]:
train = pd.read_csv(INPUT_DIR/'training_labels.csv')
submission = pd.read_csv(INPUT_DIR/'sample_submission.csv')
submission

In [ ]:
prediction_list = [
    RESULT_DIR/'pseudo_12',
    RESULT_DIR/'pseudo_seq_04',
    RESULT_DIR/'pseudo_13',
    RESULT_DIR/'pseudo_14',
    RESULT_DIR/'pseudo_seq_07',
    RESULT_DIR/'pseudo_16',
    RESULT_DIR/'pseudo_17',
    RESULT_DIR/'pseudo_18',
    RESULT_DIR/'pseudo_19',
    RESULT_DIR/'pseudo_21',
    RESULT_DIR/'pseudo_23',
    # 
    RESULT_DIR/'pseudo_24',
    RESULT_DIR/'pseudo_10',
    RESULT_DIR/'pseudo_26',
    RESULT_DIR/'pseudo_25',
    RESULT_DIR/'pseudo_07',
    RESULT_DIR/'pseudo_22',
    RESULT_DIR/'pseudo_15',
    RESULT_DIR/'pseudo_seq_03',
    RESULT_DIR/'pseudo_06',
]

In [ ]:
def generate_submission_title(predictions):
    predictions = sorted(predictions)
    name = ''
    for path in predictions:
        blocks = path.stem.split('_')
        for block in blocks:
            if block.isnumeric():
                block = int(block)
            elif block == 'pseudo':
                block = ''
            elif len(block) > 2:
                block = block[0]
            name += f'{block}'
        name += ','
    name = name[:-1]
    return name


generate_submission_title(prediction_list)

In [ ]:
preds = []
preds_full = []
oofs = []
for path in prediction_list:
    if (path/'outoffolds_tta.npy').exists():
        pred = pd.DataFrame(np.load(path/'predictions_tta.npy').squeeze(2)).transpose()
        oof = np.load(path/'outoffolds_tta.npy')
        print(f'{path.stem}: tta loaded')
    else:
        pred = pd.DataFrame(np.load(path/'predictions.npy').squeeze(2)).transpose()
        oof = np.load(path/'outoffolds.npy')
    oof_score = roc_auc_score(train['target'], oof)
    
#     plt.figure(figsize=(6, 6))
#     sns.histplot(sigmoid(pred))
#     plt.title(f'{path.stem} ({oof_score:.5f})')
#     plt.show()
    
    preds.append(pred.mean(1))
    preds_full.append(pred)
    oofs.append(pd.DataFrame(oof))
preds = pd.concat(preds, axis=1)
oofs = pd.concat(oofs, axis=1)
preds_full = np.stack(preds_full)
preds

In [ ]:
preds_full.shape

In [ ]:
preds = sigmoid(preds)
preds_full = sigmoid(preds_full)
preds

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(preds.corr())
plt.show()

In [ ]:
pred_final = preds.mean(1)

# Ridge CV

In [ ]:
X, y = sigmoid(oofs).values, train['target'].values
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=2021)
pred_final = np.zeros((5, len(submission)), dtype=np.float32)
scores = []
scaler = MinMaxScaler()
for fold, (train_idx, valid_idx) in enumerate(splitter.split(X=train, y=train['target'])):
    X_train, X_valid, y_train, y_valid = X[train_idx], X[valid_idx], y[train_idx], y[valid_idx]
    X_test = preds_full[:, :, fold].transpose()
    stack_model = Ridge(fit_intercept=False)
    stack_model.fit(X_valid, y_valid)
    y_stack = stack_model.predict(X_valid)
    rr_score = roc_auc_score(y_valid, y_stack)
    print(f'[{fold}]Stack model score: {rr_score}')
    print(f'[{fold}]Coef: {stack_model.coef_}')
    print(f'Min {y_stack.min()} / Max: {y_stack.max()}')
    print('\n')
    pred_final[fold] = scaler.fit_transform(
        stack_model.predict(X_test).reshape(-1, 1)).reshape(-1)
    scores.append(rr_score)
print(f'Ridge: {np.mean(scores)}')
pred_final = pred_final.mean(0)
# pred_final = rankdata(pred_final, axis=1).mean(0)

In [ ]:
submission['target'] = pred_final

In [ ]:
submission

In [ ]:
submission.to_csv('results/submission.csv', index=False)